## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 📈 Azure AI Agent with Bing Grounding - Financial Market Research 🏦

This notebook demonstrates how to create an Azure AI agent using **Bing Grounding** for real-time market research. Since `BingGroundingTool` is a hosted tool, it must be configured server-side on the Foundry agent — we use the `azure-ai-projects` SDK to create the agent, then `FoundryAgent` from the Agent Framework to run it.

## Features Covered:
- Creating a Foundry agent with `BingGroundingTool` via `AIProjectClient`
- Connecting to the agent with `FoundryAgent` from the Agent Framework
- Real-time financial market research using Bing search
- Proper resource cleanup

### ⚠️ Important Financial Disclaimer ⚠️
> **Financial information retrieved from web searches is for informational purposes only. Always verify with official sources and consult qualified financial advisors before making investment decisions.**

## Prerequisites

Before running this notebook, ensure you have:
1. **Azure CLI**: Installed and authenticated (`az login --use-device-code`)
2. **Microsoft Foundry project**: With deployed models
3. **Bing Connection**: Bing Grounding connection provisioned in your Foundry project
4. **Environment Variables**: Set up your `.env` file with:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
   - `GROUNDING_WITH_BING_CONNECTION_NAME`

## Import Libraries

Import the Agent Framework (`FoundryAgent`) and Azure AI Projects SDK (for creating the Bing-grounded agent server-side):

In [ ]:
import os
from pathlib import Path

# Agent Framework imports
from agent_framework.foundry import FoundryAgent

# Azure SDK imports for creating the Bing-grounded agent server-side
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)
from azure.identity import AzureCliCredential

## Initial Setup

Load environment variables from the `.env` file:

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file (two directories up)
load_dotenv('../../.env')

# Verify environment setup
endpoint = os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT')
model = os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
bing_conn_name = os.getenv('GROUNDING_WITH_BING_CONNECTION_NAME')

print("🔧 Environment Configuration:")
print(f"✅ Project Endpoint: {endpoint[:50]}..." if endpoint else "❌ AI_FOUNDRY_PROJECT_ENDPOINT not set")
print(f"✅ Model Deployment: {model}" if model else "❌ AZURE_AI_MODEL_DEPLOYMENT_NAME not set")
print(f"✅ Bing Connection: {bing_conn_name}" if bing_conn_name else "❌ GROUNDING_WITH_BING_CONNECTION_NAME not set")

## Create Bing-Grounded Agent in Foundry 🔍

Use `AIProjectClient` to create a server-side agent with `BingGroundingTool`. This agent will be accessible by name via `FoundryAgent`:

In [ ]:
credential = AzureCliCredential()

# Initialize AIProjectClient to create agent with Bing grounding
project_client = AIProjectClient(
    endpoint=endpoint,
    credential=credential,
)

# Get Bing connection ID from the Foundry project
bing_connection = project_client.connections.get(name=bing_conn_name)
conn_id = bing_connection.id
print(f"🔗 Bing Connection ID: {conn_id}")

# Create BingGroundingTool
bing_tool = BingGroundingTool(
    bing_grounding=BingGroundingSearchToolParameters(
        search_configurations=[
            BingGroundingSearchConfiguration(
                project_connection_id=conn_id
            )
        ]
    )
)

# Create the agent server-side with Bing grounding
AGENT_NAME = "financial-market-research-agent"

bing_agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a Financial Market Research Analyst with Bing search capabilities. "
            "Always: 1) Use Bing to find current interest rates, market news, and economic trends. "
            "2) Cite your sources and include relevant dates. "
            "3) Remind users that past performance doesn't guarantee future results. "
            "4) Provide disclaimers that this is not investment advice."
        ),
        tools=[bing_tool],
    ),
)

print(f"✅ Created Foundry agent: {bing_agent.name} (version: {bing_agent.version})")

## Connect with FoundryAgent and Run Queries 📈

Now use `FoundryAgent` from the Agent Framework to connect to the Bing-grounded agent and run market research queries:

In [ ]:
async def main() -> None:
    """Use FoundryAgent to connect to the Bing-grounded agent for market research."""
    
    # Connect to the server-side agent via FoundryAgent
    agent = FoundryAgent(
        agent_name=AGENT_NAME,
        project_endpoint=endpoint,
        credential=credential,
    )
    
    print("=== 📈 Azure AI Financial Market Research Agent (Bing Grounding) ===\n")
    
    # Market research query
    user_input = "What are the latest trends in the banking and financial services sector?"
    print(f"🤔 Analyst Query: {user_input}")
    response = await agent.run(user_input)
    print(f"📊 Market Research: {response.text}\n")
    print("⚠️ Disclaimer: This information is for research purposes only. Consult a financial advisor for investment decisions.")

## Execute the Example 🚀

Run the main function to see the Financial Market Research agent in action:

In [ ]:
# Run the main function
await main()

## Try Additional Market Research Queries 📊

Let's test the agent with more financial market research queries:

In [ ]:
async def financial_market_research_queries():
    """Test the agent with multiple market research queries."""
    
    agent = FoundryAgent(
        agent_name=AGENT_NAME,
        project_endpoint=endpoint,
        credential=credential,
    )
    
    queries = [
        "What are the current Federal Reserve interest rate expectations?",
        "What are the latest news about major bank earnings and financial sector performance?",
        "What is the current state of the mortgage market and housing finance trends?",
        "What are the emerging trends in fintech and digital banking?"
    ]
    
    for query in queries:
        print(f"\n{'='*60}")
        print(f"🔍 Research Query: {query}")
        print("-"*40)
        response = await agent.run(query)
        print(f"📊 Analysis: {response.text}")
        print("⚠️ For informational purposes only. Verify with official sources.")

# Run financial market research queries
await financial_market_research_queries()

## Cleanup 🧹

Optionally delete the server-side agent when done:

In [ ]:
# Uncomment to delete the agent when done
# project_client.agents.delete_version(agent_name=bing_agent.name, agent_version=bing_agent.version)
# print(f"🗑️ Deleted agent: {bing_agent.name} (version: {bing_agent.version})")

## Key Takeaways 📚

### Architecture: Two-Step Approach

Bing grounding is a **hosted tool** that must be configured server-side. The workflow is:

1. **Create the agent server-side** using `AIProjectClient` with `BingGroundingTool`
2. **Connect to it** using `FoundryAgent(agent_name=...)` from the Agent Framework

```python
# Step 1: Create agent with Bing grounding (azure-ai-projects SDK)
bing_tool = BingGroundingTool(
    bing_grounding=BingGroundingSearchToolParameters(
        search_configurations=[
            BingGroundingSearchConfiguration(project_connection_id=conn_id)
        ]
    )
)
agent_def = project_client.agents.create_version(
    agent_name="my-agent",
    definition=PromptAgentDefinition(model=model, tools=[bing_tool], instructions="...")
)

# Step 2: Run with Agent Framework
agent = FoundryAgent(agent_name="my-agent", project_endpoint=endpoint, credential=credential)
result = await agent.run("query")
```

### Why Not `FoundryChatClient` + `Agent`?

| Tool Type | Where to configure | API |
|---|---|---|
| `FunctionTool` (local functions) | In code via `tools=[...]` | `FoundryChatClient` + `Agent` |
| `BingGroundingTool` (hosted) | Server-side on Foundry agent | `FoundryAgent` |
| `FileSearchTool` (hosted) | Server-side + `get_file_search_tool()` | `FoundryChatClient` + `Agent` |
| `WebSearchTool` (Responses API) | In code via `get_web_search_tool()` | `FoundryChatClient` + `Agent` |

### Best Practices

1. **Source Citation**: Always ask the agent to cite its sources
2. **Cleanup**: Delete server-side agents when no longer needed
3. **Connection Setup**: Bing connection must be provisioned in the Foundry portal first
4. **Disclaimers**: Always include appropriate financial disclaimers

⚠️ **Disclaimer**: Web search results provide general information and should not be considered professional advice.